In [19]:
import requests
import os
import pandas as pd
import matplotlib.pyplot as plt
import random
from datetime import datetime

ROOT = ".." # Adjust to repository root

from dotenv import load_dotenv
DOTENV_PATH = os.path.join(ROOT,"../../apis/.env") # Adjust to .env file location

if load_dotenv(DOTENV_PATH):
    abs_path = os.path.abspath(DOTENV_PATH)
    drive, rel = os.path.splitdrive(abs_path)
    parts = rel.strip(os.sep).split(os.sep)
    masked = parts.copy()
    masked_parts = 1  # adjust to mask parts of your path
    max_maskable = max(0, len(parts) - 2)
    for i in range(min(masked_parts, max_maskable)):
        idx = len(parts) - 3 - i
        masked[idx] = "***"
    prefix = f"{drive}{os.sep}" if drive else (os.sep if abs_path.startswith(os.sep) else "")
    print(f"Loaded .env from {prefix}{os.sep.join(masked)}")
else:
    print("Failed to load .env file.")

API_KEY = os.getenv("AEMET_API_KEY")

def mask_token(token, unmasked_chars=3):
    return token[:unmasked_chars] + '*' * (len(token) - unmasked_chars*2) + token[-unmasked_chars:]
print(f"AEMET_API_KEY: {mask_token(API_KEY)}")

Loaded .env from c:\Users\david\***\apis\.env
AEMET_API_KEY: eyJ***********************************************************************************************************************************************************************************************************************************************************************************************************rF4


In [2]:
BASE_URL = "https://opendata.aemet.es/opendata/api"
HEADERS = {"api_key": API_KEY}

def _get_data_url(endpoint: str, params: dict | None = None) -> str | None:
    resp = requests.get(f"{BASE_URL}/{endpoint}", headers=HEADERS, params=params)
    resp.raise_for_status()
    payload = resp.json()
    return payload.get("datos")


def _download_json(url: str):
    resp = requests.get(url)
    resp.raise_for_status()
    return resp.json()

Current observations for all stations

In [3]:
datos_url = _get_data_url("observacion/convencional/todas")
observations = _download_json(datos_url)

print(len(observations))
print(observations[0])

10239
{'idema': '0009X', 'lon': 0.963335, 'fint': '2026-01-29T02:00:00+0000', 'prec': 0.0, 'alt': 406.0, 'vmax': 14.5, 'vv': 5.5, 'dv': 281.0, 'lat': 41.213892, 'dmax': 252.0, 'ubi': 'ALFORJA', 'hr': 63.0, 'tamin': 8.4, 'ta': 8.7, 'tamax': 8.7}


In [23]:
observations_df = pd.DataFrame(observations)
today_date = datetime.now().strftime("%Y%m%d")
csv_dir = os.path.join(ROOT, "csv")
os.makedirs(csv_dir, exist_ok=True)
observations_df.to_csv(os.path.join(csv_dir, f"current_observations_{today_date}.csv"), index=False)
display(observations_df.head(2))
display(observations_df.tail(2))

,idema,lon,fint,prec,alt,vmax,vv,dv,lat,dmax,...,pacutp,vvu,stdvvu,stddvu,dmaxu,tss20cm,geo925,geo850,nieve,geo700
0,0009X,0.963335,2026-01-29T02:00:00+0000,0.0,406.0,14.5,5.5,281.0,41.213892,252.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0016A,1.163611,2026-01-29T02:00:00+0000,0.0,71.0,9.3,5.4,280.0,41.145000,260.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,idema,lon,fint,prec,alt,vmax,vv,dv,lat,dmax,...,pacutp,vvu,stdvvu,stddvu,dmaxu,tss20cm,geo925,geo850,nieve,geo700
10237,C839X,-13.51021,2026-01-29T14:00:00+0000,0.0,19.0,7.6,4.4,286.0,29.229585,290.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10238,C929I,-17.88889,2026-01-29T14:00:00+0000,0.0,32.0,7.7,4.7,340.0,27.818888,350.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
random_point = random.choice(observations)
print(random_point)

{'idema': '9427X', 'lon': -1.380829, 'fint': '2026-01-28T07:00:00+0000', 'prec': 0.0, 'alt': 370.0, 'vmax': 6.1, 'vv': 1.7, 'dv': 86.0, 'lat': 41.481115, 'dmax': 265.0, 'ubi': 'LA ALMUNIA DE DOÑA GODINA', 'hr': 70.0, 'tamin': 5.6, 'ta': 5.6, 'tamax': 6.3}


Daily climatological values for a station

In [31]:
def current_observations_all_stations():
    meta = requests.get(
        f"{BASE_URL}/observacion/convencional/todas",
        headers=HEADERS,
    ).json()

    datos_url = meta["datos"]
    return requests.get(datos_url).json()


data = current_observations_all_stations()

print(len(data))
print(data[0])

9398
{'idema': '0009X', 'lon': 0.963335, 'fint': '2026-01-27T22:00:00+0000', 'prec': 0.0, 'alt': 406.0, 'vmax': 14.8, 'vv': 7.4, 'dv': 266.0, 'lat': 41.213892, 'dmax': 270.0, 'ubi': 'ALFORJA', 'hr': 82.0, 'tamin': 7.1, 'ta': 7.2, 'tamax': 7.2}


Forecasts:

In [32]:
municipality_code = "28079"  # Madrid

endpoint = f"prediccion/especifica/municipio/diaria/{municipality_code}"

datos_url = _get_data_url(endpoint)
forecast = _download_json(datos_url)

print(forecast[0]["prediccion"]["dia"][0])

{'probPrecipitacion': [{'value': 0, 'periodo': '00-24'}, {'value': 0, 'periodo': '00-12'}, {'value': 95, 'periodo': '12-24'}, {'value': 0, 'periodo': '00-06'}, {'value': 100, 'periodo': '06-12'}, {'value': 0, 'periodo': '12-18'}, {'value': 90, 'periodo': '18-24'}], 'cotaNieveProv': [{'value': '', 'periodo': '00-24'}, {'value': '', 'periodo': '00-12'}, {'value': '1200', 'periodo': '12-24'}, {'value': '', 'periodo': '00-06'}, {'value': '1000', 'periodo': '06-12'}, {'value': '', 'periodo': '12-18'}, {'value': '1300', 'periodo': '18-24'}], 'estadoCielo': [{'value': '', 'periodo': '00-24', 'descripcion': ''}, {'value': '', 'periodo': '00-12', 'descripcion': ''}, {'value': '23', 'periodo': '12-24', 'descripcion': 'Intervalos nubosos con lluvia'}, {'value': '', 'periodo': '00-06', 'descripcion': ''}, {'value': '25', 'periodo': '06-12', 'descripcion': 'Muy nuboso con lluvia'}, {'value': '17', 'periodo': '12-18', 'descripcion': 'Nubes altas'}, {'value': '45n', 'periodo': '18-24', 'descripcion':